# Capstone 9.3 — A CLI Productivity Tool

### Python Mastery · Track 9: Capstone Projects

This final capstone is the broadest synthesis in the course: a complete **command-line task manager**, built as a real multi-file project with a clean separation between logic and interface, persistent storage, a full test suite, and packaging. It draws on core Python (Tracks 1 to 3), file handling and data structures (Track 2), testing (Module 6.1), packaging (Module 6.5), and CLI design (Module 8.3).

**How to use this notebook**

- Read each section, then run the code cell beneath it. The project is built file by file with `%%writefile` into a real package layout, then run as a genuine command-line tool with `!python` and tested with `pytest`, so you see it working exactly as a user would.
- The finished project is a folder you can take away, install, and extend.
- Extension challenges with solutions appear near the end.

### What you will build

A task manager, `taskman`, with a clean architecture:

- A **storage layer** that persists tasks to a JSON file.
- A **core logic layer** (add, list, complete, remove tasks) with no input/output concerns, so it is easy to test.
- A **command-line interface** built with `argparse`, wiring user commands to the core logic.
- A **test suite** covering the core logic.
- **Packaging** so it can be installed and run as a command.

### Learning objectives

After completing this capstone you will be able to:

1. Separate core logic, storage, and interface into distinct modules.
2. Persist application state to JSON safely.
3. Build a multi-command CLI with `argparse` subcommands and exit codes.
4. Write a focused test suite for the core logic.
5. Package the tool with `pyproject.toml` so it installs as a command.

**Prerequisites:** Tracks 1 to 3, and Modules 6.1, 6.5, and 8.3.

---

## Design · Separation of Concerns

A maintainable tool separates its responsibilities so each part can change and be tested independently (the principles from Module 6.8). Our design has three layers:

- **Core logic** (`core.py`): pure functions that operate on a list of task dictionaries, adding, listing, completing, and removing. No file or screen input/output, so it is trivial to test.
- **Storage** (`storage.py`): load tasks from and save tasks to a JSON file. The only part that touches the disk.
- **Interface** (`cli.py`): parse command-line arguments and call the core logic, loading and saving through storage. The only part that talks to the user.

This layering means the core logic never needs a file or a terminal to be tested, and the storage format or interface could change without touching the logic. We build the layers from the inside out: core first, then storage, then the CLI.

In [ ]:
import os
# Create the project package directory.
os.makedirs("taskman_project/taskman", exist_ok=True)
os.makedirs("taskman_project/tests", exist_ok=True)
print("project skeleton created under taskman_project/")

## Layer 1 · Core Logic

The core operates on a plain list of task dictionaries and returns new state; it knows nothing about files or the command line. Each function does one job. Keeping it pure (input in, result out, no side effects on disk) is what makes it straightforward to test thoroughly.

In [ ]:
%%writefile taskman_project/taskman/core.py
"""Core task operations. Pure logic: no file or console input/output."""

def add_task(tasks, text):
    """Return a new task list with one task appended. Ids are assigned sequentially."""
    next_id = max((t["id"] for t in tasks), default=0) + 1
    new_task = {"id": next_id, "text": text, "done": False}
    return tasks + [new_task]

def complete_task(tasks, task_id):
    """Return a new task list with the given task marked done. Raise if it is absent."""
    if not any(t["id"] == task_id for t in tasks):
        raise KeyError(f"no task with id {task_id}")
    return [{**t, "done": True} if t["id"] == task_id else t for t in tasks]

def remove_task(tasks, task_id):
    """Return a new task list without the given task. Raise if it is absent."""
    if not any(t["id"] == task_id for t in tasks):
        raise KeyError(f"no task with id {task_id}")
    return [t for t in tasks if t["id"] != task_id]

def format_tasks(tasks, show_all=True):
    """Return a human-readable listing of tasks (pending only unless show_all)."""
    visible = tasks if show_all else [t for t in tasks if not t["done"]]
    if not visible:
        return "(no tasks)"
    lines = []
    for t in visible:
        mark = "x" if t["done"] else " "
        lines.append(f"[{mark}] {t['id']}: {t['text']}")
    return "\n".join(lines)

In [ ]:
# Exercise the core logic directly (it needs nothing else), to confirm the design.
import sys, importlib
sys.path.insert(0, "taskman_project")
from taskman import core
importlib.reload(core)

tasks = []
tasks = core.add_task(tasks, "write the capstone")
tasks = core.add_task(tasks, "review the draft")
tasks = core.complete_task(tasks, 1)
print(core.format_tasks(tasks))
print("---")
print("pending only:")
print(core.format_tasks(tasks, show_all=False))

## Layer 2 · Storage

Storage is the only layer that touches the disk. It loads tasks from a JSON file (returning an empty list if the file does not exist yet) and saves them back. Isolating persistence here means the rest of the program never deals with files, and we could switch to a database later by changing only this module.

In [ ]:
%%writefile taskman_project/taskman/storage.py
"""Persistence layer: load and save tasks as JSON. The only module that touches disk."""
import json
import os

def load_tasks(path):
    """Load tasks from a JSON file, returning an empty list if it does not exist."""
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return json.load(f)

def save_tasks(path, tasks):
    """Save tasks to a JSON file, creating parent directories if needed."""
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(path, "w") as f:
        json.dump(tasks, f, indent=2)

In [ ]:
# Confirm storage round-trips correctly: save then load returns the same data.
from taskman import storage
importlib.reload(storage)

storage.save_tasks("taskman_project/demo_tasks.json", tasks)
reloaded = storage.load_tasks("taskman_project/demo_tasks.json")
print("saved and reloaded", len(reloaded), "tasks")
print("data preserved:", reloaded == tasks)
# Loading a non-existent file yields an empty list, not an error.
print("missing file gives:", storage.load_tasks("taskman_project/does_not_exist.json"))

## Layer 3 · The Command-Line Interface

The interface ties everything together. Using `argparse` subcommands (Module 8.3), it offers `add`, `list`, `done`, and `remove` commands. Each handler loads the tasks, calls the appropriate core function, and saves the result. The CLI also returns meaningful **exit codes**: zero on success, non-zero when a command fails (for example, completing a task that does not exist).

In [ ]:
%%writefile taskman_project/taskman/cli.py
"""Command-line interface for the task manager."""
import argparse
import sys
from taskman import core, storage

DEFAULT_PATH = "tasks.json"

def build_parser():
    parser = argparse.ArgumentParser(prog="taskman", description="A simple task manager.")
    parser.add_argument("--file", default=DEFAULT_PATH, help="path to the task file")
    subs = parser.add_subparsers(dest="command", required=True)

    add_p = subs.add_parser("add", help="add a task")
    add_p.add_argument("text", help="the task description")

    list_p = subs.add_parser("list", help="list tasks")
    list_p.add_argument("--all", action="store_true", help="include completed tasks")

    done_p = subs.add_parser("done", help="mark a task complete")
    done_p.add_argument("id", type=int, help="the task id")

    remove_p = subs.add_parser("remove", help="remove a task")
    remove_p.add_argument("id", type=int, help="the task id")

    return parser

def main(argv=None):
    parser = build_parser()
    args = parser.parse_args(argv)
    tasks = storage.load_tasks(args.file)

    try:
        if args.command == "add":
            tasks = core.add_task(tasks, args.text)
            storage.save_tasks(args.file, tasks)
            print(f"added: {args.text}")
        elif args.command == "list":
            print(core.format_tasks(tasks, show_all=args.all))
        elif args.command == "done":
            tasks = core.complete_task(tasks, args.id)
            storage.save_tasks(args.file, tasks)
            print(f"completed task {args.id}")
        elif args.command == "remove":
            tasks = core.remove_task(tasks, args.id)
            storage.save_tasks(args.file, tasks)
            print(f"removed task {args.id}")
    except KeyError as e:
        print(f"error: {e}", file=sys.stderr)
        return 1                      # non-zero exit signals failure to the shell
    return 0

if __name__ == "__main__":
    sys.exit(main())

In [ ]:
# A package needs an __init__.py to be importable as a package.
import os
with open("taskman_project/taskman/__init__.py", "w") as f:
    f.write('"""taskman: a simple command-line task manager."""\n__version__ = "0.1.0"\n')
print("package marker written")
print("package files:", sorted(os.listdir("taskman_project/taskman")))

## Running the Tool

Now we use it as a real command-line program. Running `cli.py` as a module with arguments exercises the whole stack: argument parsing, core logic, and JSON persistence. We run a sequence of commands against a fresh task file and watch the state evolve, exactly as a user would at a terminal.

In [ ]:
import subprocess, sys, os

# Use a fresh task file for the demonstration.
task_file = "taskman_project/run_tasks.json"
if os.path.exists(task_file):
    os.remove(task_file)

def taskman(*args):
    """Run the CLI as a subprocess, returning (stdout, exit_code)."""
    result = subprocess.run(
        [sys.executable, "-m", "taskman.cli", "--file", "run_tasks.json", *args],
        cwd="taskman_project", capture_output=True, text=True,
    )
    return result.stdout.strip() or result.stderr.strip(), result.returncode

print(taskman("add", "write the notebook")[0])
print(taskman("add", "test the project")[0])
print(taskman("add", "package it")[0])
print("--- current list ---")
print(taskman("list")[0])

In [ ]:
# Complete a task and remove another, then list again.
print(taskman("done", "1")[0])
print(taskman("remove", "2")[0])
print("--- list including completed ---")
print(taskman("list", "--all")[0])

# Demonstrate the exit code on an error: completing a non-existent task.
output, code = taskman("done", "99")
print("--- error case ---")
print("output:", output)
print("exit code:", code, "(non-zero signals failure to the shell)")

## The Test Suite

The core logic is the part most worth testing, and because it is pure it is easy to test exhaustively without files or subprocesses (Module 6.1). We write a pytest suite covering adding, completing, removing, error cases, and formatting, then run it.

In [ ]:
%%writefile taskman_project/tests/test_core.py
import sys, os
# Make the package importable when pytest runs from the project root.
sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from taskman import core

def test_add_assigns_sequential_ids():
    tasks = core.add_task([], "first")
    tasks = core.add_task(tasks, "second")
    assert [t["id"] for t in tasks] == [1, 2]

def test_add_does_not_mutate_input():
    original = []
    core.add_task(original, "x")
    assert original == []          # add returns a new list, leaving the input untouched

def test_complete_marks_done():
    tasks = core.add_task([], "task")
    tasks = core.complete_task(tasks, 1)
    assert tasks[0]["done"] is True

def test_complete_missing_raises():
    import pytest
    with pytest.raises(KeyError):
        core.complete_task([], 99)

def test_remove_deletes_task():
    tasks = core.add_task([], "task")
    tasks = core.remove_task(tasks, 1)
    assert tasks == []

def test_remove_missing_raises():
    import pytest
    with pytest.raises(KeyError):
        core.remove_task([], 99)

def test_format_shows_pending_only():
    tasks = core.add_task([], "a")
    tasks = core.add_task(tasks, "b")
    tasks = core.complete_task(tasks, 1)
    listing = core.format_tasks(tasks, show_all=False)
    assert "a" not in listing and "b" in listing

In [ ]:
# Run the test suite against the core logic.
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q"],
    cwd="taskman_project", capture_output=True, text=True,
)
print(result.stdout)

## Packaging

Finally we make the tool installable (Module 6.5). A `pyproject.toml` describes the project and, importantly, declares a **console script entry point**, so that installing the package creates a `taskman` command on the user's path that runs our `main` function. We write the configuration and a README to complete the project.

In [ ]:
import textwrap

pyproject = textwrap.dedent("""
    [build-system]
    requires = ["setuptools>=61.0"]
    build-backend = "setuptools.build_meta"

    [project]
    name = "taskman"
    version = "0.1.0"
    description = "A simple command-line task manager."
    readme = "README.md"
    requires-python = ">=3.9"
    dependencies = []

    [project.optional-dependencies]
    dev = ["pytest"]

    [project.scripts]
    taskman = "taskman.cli:main"

    [tool.setuptools.packages.find]
    where = ["."]
    include = ["taskman*"]
""").lstrip()
with open("taskman_project/pyproject.toml", "w") as f:
    f.write(pyproject)

readme = textwrap.dedent("""
    # taskman

    A simple command-line task manager with JSON persistence.

    ## Install
    ```
    pip install -e .
    ```
    This creates a `taskman` command (via the console-script entry point).

    ## Usage
    ```
    taskman add "write the report"
    taskman list
    taskman done 1
    taskman remove 2
    taskman list --all
    ```

    ## Architecture
    - `taskman/core.py`    - pure task logic (no input/output), fully tested
    - `taskman/storage.py` - JSON load/save (the only disk access)
    - `taskman/cli.py`     - argparse command-line interface and entry point

    ## Develop
    ```
    pip install -e ".[dev]"
    pytest
    ```
""").lstrip()
with open("taskman_project/README.md", "w") as f:
    f.write(readme)

import os
print("final project layout:")
for root, dirs, files in os.walk("taskman_project"):
    dirs[:] = [d for d in dirs if d not in ("__pycache__", ".pytest_cache")]
    for name in sorted(files):
        if name.endswith((".json",)) and "tasks" in name:
            continue
        print("  " + os.path.join(root, name).replace("taskman_project/", ""))

In [ ]:
# Confirm the entry point is wired correctly by installing the package and running the command.
import subprocess, sys
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "--quiet", "--break-system-packages"],
    cwd="taskman_project", capture_output=True, text=True,
)
print("install return code:", install.returncode)

# Now the 'taskman' command exists; run it via the installed entry point.
run = subprocess.run(["taskman", "--file", "/tmp/entry_tasks.json", "add", "installed and working"],
                     capture_output=True, text=True)
print("entry-point command output:", run.stdout.strip())
listing = subprocess.run(["taskman", "--file", "/tmp/entry_tasks.json", "list"],
                         capture_output=True, text=True)
print("list via installed command:", listing.stdout.strip())

The tool is now a genuine installed command: `pip install -e .` created a `taskman` executable that runs our code, exactly as a published package would. Every layer, core logic, storage, CLI, tests, and packaging, came together into a working, installable application.

---

## Demonstrated Variations

### Variation 1 — Core logic composes cleanly

In [ ]:
from taskman import core
# Because the core is pure, operations chain naturally and predictably.
tasks = core.add_task([], "a")
tasks = core.add_task(tasks, "b")
tasks = core.add_task(tasks, "c")
tasks = core.complete_task(tasks, 2)
tasks = core.remove_task(tasks, 1)
print(core.format_tasks(tasks))

### Variation 2 — The CLI accepts a custom file path

In [ ]:
out, _ = taskman("add", "task in a custom file")
print(out)
# Each invocation used --file run_tasks.json; pointing at a different file keeps separate lists.
out, _ = taskman("list")
print(out)

### Variation 3 — Listing an empty set is handled

In [ ]:
import subprocess, sys
empty = subprocess.run(
    [sys.executable, "-m", "taskman.cli", "--file", "empty_demo.json", "list"],
    cwd="taskman_project", capture_output=True, text=True,
)
print("empty list output:", empty.stdout.strip())

---

## Extension Challenges

**Challenge 1.** Add a `clear` subcommand to the CLI that removes all completed tasks, saving the result. (You may add a core function `clear_completed(tasks)` and test it.)

In [ ]:
# Your solution here


**Challenge 2.** Add a priority field to tasks: extend `add_task` to accept an optional priority (default "normal"), and a core function that returns tasks sorted with "high" priority first. Demonstrate it.

In [ ]:
# Your solution here


**Challenge 3.** Write a test that exercises the storage layer: save a list of tasks to a temporary file, load it back, and assert the data is identical. (Use a path under `/tmp`.)

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
from taskman import core

def clear_completed(tasks):
    """Return only the tasks that are not done."""
    return [t for t in tasks if not t["done"]]

# Demonstrate, and verify it integrates with the rest of the core.
tasks = core.add_task([], "keep me")
tasks = core.add_task(tasks, "finish me")
tasks = core.complete_task(tasks, 2)
remaining = clear_completed(tasks)
print("after clearing completed:", core.format_tasks(remaining))
# In the CLI, a 'clear' subcommand would call clear_completed and save the result.

**Solution 2**

In [ ]:
def add_task_with_priority(tasks, text, priority="normal"):
    next_id = max((t["id"] for t in tasks), default=0) + 1
    return tasks + [{"id": next_id, "text": text, "done": False, "priority": priority}]

def by_priority(tasks):
    order = {"high": 0, "normal": 1, "low": 2}
    return sorted(tasks, key=lambda t: order.get(t.get("priority", "normal"), 1))

tasks = add_task_with_priority([], "low one", "low")
tasks = add_task_with_priority(tasks, "urgent one", "high")
tasks = add_task_with_priority(tasks, "normal one")
for t in by_priority(tasks):
    print(f"{t['priority']:6} - {t['text']}")

**Solution 3**

In [ ]:
import sys
sys.path.insert(0, "taskman_project")
from taskman import storage

def test_storage_round_trip():
    path = "/tmp/taskman_roundtrip.json"
    data = [{"id": 1, "text": "x", "done": False}, {"id": 2, "text": "y", "done": True}]
    storage.save_tasks(path, data)
    assert storage.load_tasks(path) == data

test_storage_round_trip()
print("storage round-trip test passed")

---

## Summary and Key Points

- A maintainable CLI tool separates concerns into layers: pure core logic (no input/output), a storage layer (the only disk access), and an interface layer (the only user contact); each can change and be tested independently.
- Keeping the core pure, functions that take state and return new state, makes it trivial to test exhaustively and lets operations compose predictably.
- The `argparse` interface uses subcommands for `add`, `list`, `done`, and `remove`, loads and saves through the storage layer, and returns exit codes (zero for success, non-zero for failure) that shells can act on.
- The test suite targets the core logic directly, covering normal cases, error cases (missing ids raising `KeyError`), and formatting, without needing files or subprocesses.
- Packaging with `pyproject.toml` and a console-script entry point (`taskman = "taskman.cli:main"`) turns the project into an installable command, the same path any published tool follows.

### Track 9 complete

This concludes Track 9, and the course. Across three capstones you have integrated the whole curriculum: a data pipeline (files, cleaning, databases, reporting, testing), an async API aggregator (concurrency, web requests, resilience patterns), and a command-line productivity tool (architecture, persistence, CLI design, testing, packaging). Each is a real, runnable project rather than an exercise, and together they demonstrate how the individual skills from Tracks 1 to 8 combine into working software.